# 1.1.1 智能医疗系统中的业务数据处理流程设计

## 任务说明
某医疗机构计划引入智能医疗系统，以提升诊断效率和准确性。通过分析患者的历史数据，使用机器学习算法预测患者的健康风险，从而辅助医生进行诊断和治疗。为此，该机构需要设计一套全面的业务数据处理流程，确保数据处理的高效性和准确性，为人工智能模型提供可靠的输入数据。

In [10]:
import pandas as pd
import numpy as np

# 读取数据集
data = pd.read_csv('data/patient_data.csv')
data.head()

,PatientID,Age,BMI,BloodPressure,Cholesterol,DaysInHospital
0,P001,45,28.5,140,220,12
1,P002,32,22.1,120,190,5
2,P003,67,26.8,150,240,9
3,P004,28,19.7,110,180,3
4,P005,55,31.2,160,260,15


## 任务1：统计住院天数超过7天的患者数量及其占比

In [17]:
# 创建新列'RiskLevel'，根据住院天数判断风险等级
# data['RiskLevel'] = np.where(data['DaysInHospital'] > 7, '高风险患者', '低风险患者')
data['RiskLevel'] =  np.where(data['DaysInHospital'] > 7, '高风险患者', '低风险患者')
#print(data['RiskLevel'].head())
print(data.head())

# 统计不同风险等级的患者数量
risk_counts = data['RiskLevel'].value_counts()
print(risk_counts)

# 计算高风险患者占比
high_risk_ratio = risk_counts['高风险患者'] / len(data)

# 计算低风险患者占比
low_risk_ratio = risk_counts['低风险患者'] / len(data)

# 输出结果
print("高风险患者数量:", risk_counts['高风险患者'])
print("低风险患者数量:", risk_counts['低风险患者'])
print("高风险患者占比:", high_risk_ratio)
print("低风险患者占比:", low_risk_ratio)

  PatientID  Age   BMI  BloodPressure  Cholesterol  DaysInHospital RiskLevel  \
0      P001   45  28.5            140          220              12     高风险患者   
1      P002   32  22.1            120          190               5     低风险患者   
2      P003   67  26.8            150          240               9     高风险患者   
3      P004   28  19.7            110          180               3     低风险患者   
4      P005   55  31.2            160          260              15     高风险患者   

  BMIRange  
0       肥胖  
1       正常  
2       超重  
3       正常  
4       肥胖  
RiskLevel
高风险患者    16
低风险患者    14
Name: count, dtype: int64
高风险患者数量: 16
低风险患者数量: 14
高风险患者占比: 0.5333333333333333
低风险患者占比: 0.4666666666666667


## 任务2：统计不同BMI区间中高风险患者的比例和统计不同BMI区间中的患者数

In [27]:
# ============================================
# 任务2：BMI区间分析 - 统计各BMI区间高风险患者比例和患者数量
# 考试分值：4分(pd.cut) + 2分(groupby比例) + 1分(计数) = 7分
# ============================================

# --------------------------------------------------
# 步骤1: 定义BMI区间边界和对应标签
# --------------------------------------------------
# bins: 区间边界点，5个数字定义4个区间
#   [0, 18.5)     → 偏瘦  (0 ≤ BMI < 18.5)
#   [18.5, 24)    → 正常  (18.5 ≤ BMI < 24)  
#   [24, 28)      → 超重  (24 ≤ BMI < 28)
#   [28, np.inf)  → 肥胖  (28 ≤ BMI，无限大)
# 
# np.inf: 正无穷大，表示区间无上限
# 中国标准BMI分类：偏瘦<18.5，正常18.5-23.9，超重24-27.9，肥胖≥28
# --------------------------------------------------
bmi_bins = [0, 18.5, 24, 28, np.inf]
bmi_labels = ['偏瘦', '正常', '超重', '肥胖']

# --------------------------------------------------
# 步骤2: 使用 pd.cut 进行区间划分 (核心考点！)
# --------------------------------------------------
# pd.cut() 参数说明：
#   x: 要分箱的数据（BMI列）
#   bins: 区间边界数组
#   labels: 每个区间的标签名称（必须与区间数相同）
#   right=False: 左闭右开区间 [a, b)，考试必须写！
#
# 结果：在data中新增'BMIRange'列，每行显示对应的BMI分类
# --------------------------------------------------
data['BMIRange'] = pd.cut(data['BMI'], bins=bmi_bins, labels=bmi_labels, right=False)

# --------------------------------------------------
# 步骤3: 分组统计高风险患者比例（难点！）
# --------------------------------------------------
# data.groupby('BMIRange'): 按BMI区间分组
# ['RiskLevel']: 选取风险等级列
# .apply(lambda x: ...): 对每个分组应用自定义函数
#   (x == '高风险患者'): 返回True/False布尔数组
#   .mean(): 计算True的比例（True=1, False=0）
#
# 结果示例：肥胖组11人全是高风险 → 比例1.0
#          超重组8人有5人高风险 → 比例0.625
# --------------------------------------------------
bmi_risk_rate = data.groupby('BMIRange')['RiskLevel'].apply(
    lambda x: (x == '高风险患者').mean())

# --------------------------------------------------
# 【拓展】分组后查看数据的两种方式对比
# --------------------------------------------------
# 方式A: .head(2) - 扁平合并（按原索引排序，看起来混在一起）
#   data[['BMIRange','RiskLevel','BMI']].groupby('BMIRange').head(2)
#   输出：0 肥胖... 1 正常... 2 超重...（按原行号0,1,2排序）
#
# 方式B: .apply(lambda x: x.head(2)) - 分组显示（带层级索引）
#   data.groupby('BMIRange').apply(lambda x: x[['RiskLevel','BMI']].head(2))
#   输出：偏瘦  9 ...
#        正常  1 ...
#             3 ...  （按分组名+原行号两层排序）
#
# 原理：.apply让每个分组"各自独立处理"，pandas再"贴标签"组合结果
#      所以能看到层级分明的分组显示（MultiIndex）
# --------------------------------------------------

# --------------------------------------------------
# 步骤4: 统计每个BMI区间的患者总数
# --------------------------------------------------
# value_counts(): 统计每个分类出现的次数
# 默认按数量降序排列
# --------------------------------------------------
bmi_patient_count = data['BMIRange'].value_counts()

# --------------------------------------------------
# 输出结果
# --------------------------------------------------
print("BMI区间中高风险患者的比例和患者数:")
print("\n【高风险患者比例】")
print(bmi_risk_rate) 
print("\n【各区间患者数量】")
print(bmi_patient_count)

BMI区间中高风险患者的比例和患者数:

【高风险患者比例】
BMIRange
偏瘦    0.000
正常    0.000
超重    0.625
肥胖    1.000
Name: RiskLevel, dtype: float64

【各区间患者数量】
BMIRange
肥胖    11
正常    10
超重     8
偏瘦     1
Name: count, dtype: int64


/var/folders/kj/pr1vfg4j7tj53f25x7zqkfph0000gp/T/ipykernel_67502/337977967.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bmi_risk_rate = data.groupby('BMIRange')['RiskLevel'].apply(


## 任务3：统计不同年龄区间中高风险患者的比例和统计不同年龄区间中的患者数

In [ ]:
# ============================================
# 任务3：年龄区间分析 - 统计各年龄区间高风险患者比例和患者数量
# 考试分值：4分(pd.cut) + 2分(groupby比例) + 1分(计数) = 7分
# ============================================

# --------------------------------------------------
# 步骤1: 定义年龄区间边界和对应标签
# --------------------------------------------------
# bins: 7个数字定义6个年龄段
#   [0, 26)     → ≤25岁    (0 ≤ 年龄 < 26，实际为1-25岁)
#   [26, 36)    → 26-35岁  (26 ≤ 年龄 < 36)
#   [36, 46)    → 36-45岁  (36 ≤ 年龄 < 46)
#   [46, 56)    → 46-55岁  (46 ≤ 年龄 < 56)
#   [56, 66)    → 56-65岁  (56 ≤ 年龄 < 66)
#   [66, np.inf)→ ＞65岁   (66 ≤ 年龄，无限大)
#
# 注意：bins数量 = 区间数 + 1
#      labels数量 = 区间数
# --------------------------------------------------
age_bins = [0, 26, 36, 46, 56, 66, np.inf]
age_labels = ['≤25岁', '26-35岁', '36-45岁', '46-55岁', '56-65岁', '＞65岁']

# --------------------------------------------------
# 步骤2: 使用 pd.cut 进行区间划分（同BMI任务逻辑）
# --------------------------------------------------
# 在data中新增'AgeRange'列存储年龄分类
# right=False 确保 26岁整归为"26-35岁"而非"≤25岁"
# --------------------------------------------------
data['AgeRange'] = pd.cut(data['Age'], bins=age_bins, labels=age_labels, right=False)

# --------------------------------------------------
# 步骤3: 分组统计高风险患者比例
# --------------------------------------------------
# 逻辑同任务2：按年龄区间分组，计算每组高风险比例
# 结果解读：
#   ≤25岁: NaN（本数据集无此年龄段患者）
#   26-35岁: 0.0（年轻患者8人，无高风险）
#   46-55岁: 0.833（该年龄段高风险占比最高）
#   56-65岁, ＞65岁: 1.0（中老年全部高风险）
# --------------------------------------------------
age_risk_rate = data.groupby('AgeRange')['RiskLevel'].apply(
    lambda x: (x == '高风险患者').mean())

# --------------------------------------------------
# 【拓展】lambda x 和 .apply() 详解
# --------------------------------------------------
# lambda x: 表达式  =  即插即用的小函数，不需要def定义
#   x: 输入参数（这里是分组后的RiskLevel列）
#   :  分隔符，左边参数，右边返回值
#   (x == '高风险患者').mean(): 函数体，返回比例
#
# 执行流程：
#   分组1 → lambda(x) → 计算比例 → 0.0
#   分组2 → lambda(x) → 计算比例 → 0.625
#   ...
# .apply()让每个分组"各自独立处理"，再用MultiIndex组合结果
# 所以能看到层级分明的分组显示
# --------------------------------------------------

# --------------------------------------------------
# 步骤4: 统计每个年龄区间的患者总数
# --------------------------------------------------
# 注意：NaN表示该区间无数据，不是0
# --------------------------------------------------
age_patient_count = data['AgeRange'].value_counts()

# --------------------------------------------------
# 输出结果
# --------------------------------------------------
print("年龄区间中高风险患者的比例和患者数:")
print("\n【高风险患者比例】")
print(age_risk_rate) 
print("\n【各区间患者数量】")
print(age_patient_count)

# --------------------------------------------------
# 📌 考试要点总结
# --------------------------------------------------
# 1. pd.cut 必须加 right=False（左闭右开）
# 2. bins 最后一个值用 np.inf 表示无限大
# 3. groupby().apply(lambda) 是计算比例的固定写法
# 4. (x == '高风险患者').mean() 中 mean()对布尔值求平均
# --------------------------------------------------

年龄区间中高风险患者的比例和患者数:
AgeRange
≤25岁           NaN
26-35岁    0.000000
36-45岁    0.285714
46-55岁    0.833333
56-65岁    1.000000
＞65岁      1.000000
Name: RiskLevel, dtype: float64
AgeRange
26-35岁    8
36-45岁    7
46-55岁    6
＞65岁      5
56-65岁    4
≤25岁      0
Name: count, dtype: int64


/var/folders/kj/pr1vfg4j7tj53f25x7zqkfph0000gp/T/ipykernel_67502/879862672.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_risk_rate = data.groupby('AgeRange')['RiskLevel'].apply(lambda x: (x == '高风险患者').mean())
